# NumPilot 全新机器源码验收

本 Notebook 不使用 wheelhouse。只克隆 NumPilot，AsNumpy 从固定官方源码现场编译；CANNBot 仅在 gap/operator lane 中从固定提交拉取并校验，direct migration 不依赖它。NumPilot 锁定 profile 与独立审计的官方最新提交中，公开 `matmul` 都是 CPU fallback；本轮若接受它，只验证兼容性，另行授权后才做数值实跑，不发布 NPU 加速比。

> 发布状态（2026-08-27）：框架 hardening 集成候选尚无远端不可变 tag，`v0.2.0a2` 只是基线且不包含本文合同。全新机器必须等待发布说明给出的新不可变 tag。

In [ ]:
from pathlib import Path
import subprocess

REPO = Path.cwd().resolve()
assert (REPO / 'install.sh').is_file(), '请先进入刚克隆的 NumPilot 仓库'
PERSISTENT_ROOT = REPO.parent
ACTIVATION = PERSISTENT_ROOT / 'activate-numpilot.sh'
print(REPO)


## 显式同意联网；按 lane 选择 CANNBot

direct lane 保持 `WITH_CANNBOT=False`，不要求 CANNBot 许可证。只有 gap/operator lane 才先审阅固定 commit `1ef6372a538f39b0b7ed34666a3e1f465fb4936a` 的 CANN Open Software License v2.0，并同时设置 `WITH_CANNBOT=True`、`ACCEPT_CANNBOT_LICENSE=True`。若默认私有 Python 镜像缺包，由用户显式向安装命令附加 `--pip-index-url https://pypi.org/simple`；安装器不会悄悄换源。

In [ ]:
ALLOW_NETWORK = False
WITH_CANNBOT = False
ACCEPT_CANNBOT_LICENSE = False
assert ALLOW_NETWORK, '未允许安装器联网，停止'
install_args = [
    str(REPO / 'install.sh'),
    '--allow-network',
    '--persistent-root', str(PERSISTENT_ROOT),
]
if WITH_CANNBOT:
    assert ACCEPT_CANNBOT_LICENSE, '未接受 CANNBot 许可证，停止'
    install_args.extend(['--with-cannbot', '--accept-cannbot-license'])
subprocess.run(install_args, cwd=REPO, check=True)


## 离线复验

下列命令先 source 安装器生成的无凭据 activation（重复 source 必须幂等），再使用仓库 `.venv` 验证 AsNumpy README smoke 与 NPU 环境；只有 `WITH_CANNBOT=True` 时才验证 CANNBot Skill 清单。

In [ ]:
PYTHON = REPO / '.venv' / 'bin' / 'python'
def activated(*arguments):
    shell = '. "$1"; . "$1"; shift; exec "$@"'
    subprocess.run([
        'bash', '--noprofile', '--norc', '-c', shell, 'numpilot',
        str(ACTIVATION), str(PYTHON), *arguments,
    ], check=True)
activated('-m', 'numpilot', '--version')
verify_args = ['-m', 'numpilot', 'bootstrap', '--verify']
if WITH_CANNBOT:
    verify_args.extend(['--with-cannbot', '--accept-cannbot-license'])
activated(*verify_args)
activated('-m', 'numpilot', 'doctor', '--backend', 'npu')


## Configured model 与迁移停点

克隆不会生成或复制 `llm.json`。先 source activation，再运行 `python -m numpilot configure-llm --init`，填写新机器自己的 endpoint、Token、精确 model、provider 审计标签和 `protocol=openai-chat-completions`，再执行 `--check` 和 `--probe`。响应 model 默认精确匹配；alias 必须显式 allowlist。GLM-5.3/provider=glm 只是配置示例。代理只在需要时从当前进程注入，不写入仓库。

上传待测纯 NumPy 文件后先报告安装与能力分类；在用户明确说“开始迁移”前不要调用模型。Case01 必须由用户显式接受 `matmul` fallback，结果固定 `strict_npu=false`、`performance_valid=false`，且第一轮不得继续 Case02–10。